# ENSANUT 2018-19 — Data integration
Nullable join keys and reproducible overlap audits.

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from tqdm.auto import tqdm
DATA_DIR=Path('../../data')
KEYS_IND=['UPM','VIV_SEL','HOGAR','NUMREN']
KEYS_HOG=['UPM','VIV_SEL','HOGAR']

def load_ensanut_csv(path):
    for enc in ('utf-8-sig','latin-1','cp1252'):
        try:
            d=pd.read_csv(path,sep=';',encoding=enc,low_memory=False)
            d.columns=[re.sub(r'[^A-Za-z0-9_]','',str(c)).upper() for c in d.columns]
            if 'UPM' in d.columns: return d
        except (UnicodeDecodeError,pd.errors.ParserError): pass
    raise ValueError(f'Could not load {path}')

def harmonize_keys(df,keys):
    d=df.copy()
    for k in keys:
        if k in d:
            v=pd.to_numeric(d[k].astype('string').str.replace(',','.',regex=False).str.strip(),errors='coerce')
            if (v.notna()&(v.round()!=v)).any(): print(f'WARNING: non-integer {k}: {(v.notna()&(v.round()!=v)).sum()}')
            d[k]=v.round().astype('Int64')
    return d

def audit_keys(left,right,keys,left_name,right_name):
    l=left.dropna(subset=keys).drop_duplicates(keys); r=right.dropna(subset=keys).drop_duplicates(keys)
    ls=set(map(tuple,l[keys].to_numpy())); rs=set(map(tuple,r[keys].to_numpy())); inter=ls&rs
    out={'left':left_name,'right':right_name,'keys':'|'.join(keys),'left_unique':len(ls),'right_unique':len(rs),'intersection':len(inter),'left_match_rate':len(inter)/len(ls) if ls else np.nan,'left_missing_rows':int(left[keys].isna().any(axis=1).sum()),'right_missing_rows':int(right[keys].isna().any(axis=1).sum())}
    print(out); return out

def merge_module(left,right,keys,label,audits):
    audits.append(audit_keys(left,right,keys,'Adults',label))
    if right.duplicated(keys).any(): raise ValueError(f'{label} has duplicate join keys')
    overlap=[c for c in right if c in left and c not in keys]
    return left.merge(right.drop(columns=overlap),on=keys,how='left',validate='one_to_one')


In [2]:
modules={'Adults':'CS_ADULTOS.csv','Anthropometry':'CN_ANTROPOMETRIA.csv','Residents':'CS_RESIDENTES.csv','Households':'CS_HOGARES.csv','Biochemistry':'E18_MUESAN_DETBIO_ADUL_20200413.csv'}
loaded={n:harmonize_keys(load_ensanut_csv(DATA_DIR/f),KEYS_IND+KEYS_HOG) for n,f in tqdm(modules.items())}
audits=[]; df_final=loaded['Adults'].copy(); ant=loaded['Anthropometry'].copy()
ant['WEIGHT_FINAL']=pd.to_numeric(ant.get('PESO1_1'),errors='coerce').combine_first(pd.to_numeric(ant.get('PESO12_1'),errors='coerce'))
ant['HEIGHT_FINAL']=pd.to_numeric(ant.get('TALLA4_1'),errors='coerce').combine_first(pd.to_numeric(ant.get('TALLA15_1'),errors='coerce'))
df_final=merge_module(df_final,ant,KEYS_IND,'Anthropometry',audits)
for n,k in [('Residents',KEYS_IND),('Biochemistry',KEYS_IND),('Households',KEYS_HOG)]: df_final=merge_module(df_final,loaded[n],k,n,audits)
pd.DataFrame(audits).to_csv(DATA_DIR/'integration_key_audit.csv',index=False)
valid=df_final[['WEIGHT_FINAL','HEIGHT_FINAL']].notna().all(axis=1); print(f'Complete anthropometry: {valid.sum():,}/{len(df_final):,} ({valid.mean():.1%})')
prov=[]
for n,d in [('Adults',loaded['Adults']),('Anthropometry',ant),('Residents',loaded['Residents']),('Biochemistry',loaded['Biochemistry']),('Households',loaded['Households'])]:
    prov += [{'column':c,'module':n} for c in d.columns]
pd.DataFrame(prov).drop_duplicates('column').to_csv(DATA_DIR/'column_provenance.csv',index=False)
df_final.to_csv(DATA_DIR/'ensanut_integrated.csv',index=False)


  0%|          | 0/5 [00:00<?, ?it/s]

{'left': 'Adults', 'right': 'Anthropometry', 'keys': 'UPM|VIV_SEL|HOGAR|NUMREN', 'left_unique': 43070, 'right_unique': 33818, 'intersection': 17355, 'left_match_rate': 0.4029486881820293, 'left_missing_rows': 0, 'right_missing_rows': 0}
{'left': 'Adults', 'right': 'Residents', 'keys': 'UPM|VIV_SEL|HOGAR|NUMREN', 'left_unique': 43070, 'right_unique': 158044, 'intersection': 43070, 'left_match_rate': 1.0, 'left_missing_rows': 0, 'right_missing_rows': 0}
{'left': 'Adults', 'right': 'Biochemistry', 'keys': 'UPM|VIV_SEL|HOGAR|NUMREN', 'left_unique': 43070, 'right_unique': 2905, 'intersection': 2891, 'left_match_rate': 0.06712328767123288, 'left_missing_rows': 0, 'right_missing_rows': 0}
{'left': 'Adults', 'right': 'Households', 'keys': 'UPM|VIV_SEL|HOGAR', 'left_unique': 43070, 'right_unique': 44612, 'intersection': 43070, 'left_match_rate': 1.0, 'left_missing_rows': 0, 'right_missing_rows': 0}
Complete anthropometry: 17,164/43,070 (39.9%)
